# Plot

In [ ]:
library(readr)
library(purrr)
library(dplyr)
library(stringr)
library(fuzzyjoin)
library(ggplot2)
library(dplyr)
library(tidyverse)
library(Matrix)
library(reshape2)
library(RColorBrewer)

In [ ]:
options(repr.matrix.max.cols = Inf,  # show all columns
        repr.matrix.max.rows = 200)  # adjust rows as you like

## 0. Plot parameters

In [ ]:
out_dir = "/ceph.groups/mshahbazi.grp/rsakata/HE_EXP186/cellpose/output/plots"
root_dir = "/ceph.groups/mshahbazi.grp/rsakata/HE_EXP186/cellpose/output/1_cellpose"
sample_sheet_csv = "/ceph.groups/mshahbazi.grp/rsakata/HE_EXP186/sample_sheet.csv"


In [ ]:
EXP = "EXP186"
#EXP_SUB = "T2"

In [ ]:
FONT.SIZE <- 7
LABEL.FONT.SIZE <- 7
w <- 2 
h <- 2.5
LINE.W <- 0.5/2.141959

# Set geom defaults globally
update_geom_defaults("line",      list(linewidth = LINE.W))
update_geom_defaults("errorbar",  list(linewidth = LINE.W))
#update_geom_defaults("point",     list(size = LINE.W, stroke = LINE.W))

settheme <- theme_minimal() + 
  theme(
    text = element_text(family = "sans"), 
    panel.background = element_blank(),
    panel.grid.major = element_blank(), 
    panel.grid.minor = element_blank(),
    plot.background = element_blank(),
    axis.ticks = element_line(colour = "black", linewidth = LINE.W),
    axis.ticks.length = unit(0.1, "cm"), 
    axis.line = element_line(linewidth = LINE.W, colour = "black"),
    axis.title = element_text(size = FONT.SIZE),
    axis.text = element_text(colour = "black", size = FONT.SIZE),
    strip.text = element_text(size = FONT.SIZE), 
    strip.text.y.left = element_text(angle = 0, hjust = 1, size = FONT.SIZE),
    legend.position = "right",
    legend.title = element_text(size = FONT.SIZE), 
    legend.text = element_text(size = FONT.SIZE),
    legend.key.size = unit(0.3, "cm"),
    axis.text.x = element_text(colour = "black", angle = 0, size = LABEL.FONT.SIZE),
    title = element_text(size = FONT.SIZE) 
  )


In [ ]:
col_aneu = c("euploid"= "#D4D1B3","monosomy"="#109E9D","trisomy"="#F26B3B","complex"= "#886DB0")

col_condition_2 = c("control" = "#285F62", 
               "reversine" = "#CA4F33", 
               "mosaic"= "#E2A557")

col_sample_name = c("G_R"= "#5E5E5E","Grev_R"="#86AB30","Rrev_G"="#EB5951", "Grev_Rrev"="#F0A329")

col_GFP_condition = c("pos"= "#86AB30","neg"="#8d8d8dff")

col_RFP_condition = c("pos"= "#EB5951","neg"="#8d8d8dff")


col_GATA3 = "#489C9C"
col_NANOG = "#EA9542"
col_neg    = "#8d8d8dff"

col_GFP = "#86AB30"
col_RFP = "#EB5951"

col_condition = c("developed" = "#285F62", 
               "failed" = "#CA4F33")

## 1. Extract summary files

In [ ]:
if (!dir.exists(out_dir)) { 
  dir.create(out_dir, recursive = TRUE, showWarnings = FALSE)
 }

In [ ]:
# Find matching CSVs in all subfolders
csv_paths <- list.files(
  path = root_dir,
  pattern = "\\.csv$",
  recursive = TRUE,
  full.names = TRUE
)

if (length(csv_paths) == 0) stop("No matching files found.")

# Read and bind
combined_df <- purrr::map_dfr(csv_paths, ~ readr::read_csv(.x, show_col_types = FALSE))


#rename colums and add new column for sample number
combined_df <- combined_df %>%
  rename(image = sample) %>%
  mutate(sample = str_replace(image, "_.*$", ""))

combined_df$sample <- as.character(combined_df$sample)


In [ ]:
head(combined_df)

In [ ]:
#merge with sample sheet
sample_sheet <- read_csv(sample_sheet_csv, show_col_types = FALSE)
sample_sheet$sample <- as.character(sample_sheet$sample)

merged_df <- sample_sheet %>%
  left_join(combined_df, by = "sample")   # keeps all rows from df1

In [ ]:
sample_sheet <- read_csv(sample_sheet_csv, show_col_types = FALSE)
sample_sheet

In [ ]:
head(merged_df)

In [ ]:
merged_df$exp = EXP
#merged_df$exp_sub = EXP_SUB

In [ ]:
order_cond <- c("developed","failed")

merged_df <- merged_df %>%
  mutate(condition = factor(condition, levels = order_cond))


order_cond <- c("MS967", "MS1000", "MS989", "MS744", "MS811", "MS1024", "MS757", "MS763", "MS764", "MS496", "MS330", "MS776")

merged_df <- merged_df %>%
  mutate(sample_name = factor(sample, levels = order_cond))

#   Experiment,sample,condition,karyotype,complex
# EXP186,MS967,developed,19+,mono/tri
# EXP186,MS1000,developed,19+,mono/tri
# EXP186,MS989,developed,19+,mono/tri
# EXP186,MS744,developed,"12+, 14- ",complex
# EXP186,MS811,developed,"15+, 18+",complex
# EXP186,MS1024,developed,"8+, 22`=",complex
# EXP186,MS757,failed,22-,mono/tri
# EXP186,MS763,failed,22-,mono/tri
# EXP186,MS764,failed,22-,mono/tri
# EXP186,MS496,failed,"16-, 19-",complex
# EXP186,MS330,failed,"15-, 21-",complex
# EXP186,MS776,failed,"22-, 3+",complex

## 2. Preprocess

In [ ]:
colnames(merged_df)

In [ ]:
head(merged_df)

### Normalise intensities

In [ ]:
title = "GATA3_dapi"

w <- 3.5
h <- 2.5
options(repr.plot.width=w, repr.plot.height=h)

GATA3_thresh = 0.6
GATA3_thresh2 = 20

ggscatter = ggplot(merged_df, aes(x = Mean_dapi, y = Mean_GATA3, color = image)) +
  geom_point(alpha = 0.6, size = 0.6) +
  labs(x = " Mean_dapi", y = " Mean_GATA3", title = "") +
  settheme+
  #scale_color_manual(values=col_sample_name)+
  geom_abline(slope = GATA3_thresh, intercept = 0, linewidth = 0.7, linetype = "dashed", color = "gray0") +
  geom_hline(yintercept = GATA3_thresh2, linetype = "dashed", color = "grey0", linewidth = 0.7)
ggsave(plot = ggscatter, filename = sprintf("%s/%s.pdf",out_dir, title), w = w, h = h)
ggscatter

In [ ]:
title = "NANOG_dapi"

w <- 3.5
h <- 2.5
options(repr.plot.width=w, repr.plot.height=h)


line_slope <- 0.9   # -0.6
line_intercept <- 30


ggscatter = ggplot(merged_df, aes(x = Mean_dapi, y = Mean_NANOG, color = image)) +
  geom_point(alpha = 0.6, size = 0.6) +
  labs(x = " Mean_dapi", y = " Mean_NANOG", title = "") +
  settheme +
  geom_abline(slope = line_slope, intercept = line_intercept, linetype = "dashed") 
  #scale_color_manual(values=col_sample_name)
ggsave(plot = ggscatter, filename = sprintf("%s/%s.pdf",out_dir, title), w = w, h = h)
ggscatter

In [ ]:
title = "GATA4_dapi"

w <- 3.5
h <- 2.5
options(repr.plot.width=w, repr.plot.height=h)

GATA4_thresh = 0.5

ggscatter = ggplot(merged_df, aes(x = Mean_dapi, y = Mean_GATA4, color = image)) +
  geom_point(alpha = 0.6, size = 0.6) +
  labs(x = " Mean_dapi", y = " Mean_GATA4", title = "") +
  settheme+
  geom_abline(slope = GATA4_thresh, intercept = 0, linewidth = 0.7, linetype = "dashed", color = "gray0") 
  #scale_color_manual(values=col_sample_name)
ggsave(plot = ggscatter, filename = sprintf("%s/%s.pdf",out_dir, title), w = w, h = h)
ggscatter

In [ ]:
#Compute the normalised intensities
#merged_df$GFP_norm = merged_df$Mean_GFP/merged_df$Mean_dapi
#merged_df$GFP_norm = merged_df$Mean_GFP/merged_df$Mean_dapi
merged_df$GATA3_norm = merged_df$Mean_GATA3/merged_df$Mean_dapi
merged_df$NANOG_norm = merged_df$Mean_NANOG/merged_df$Mean_dapi
merged_df$GATA4_norm = merged_df$Mean_GATA4/merged_df$Mean_dapi
#merged_df$caspase3_norm = merged_df$Mean_caspase3/merged_df$Mean_dapi


In [ ]:
order_cond <- c("developed","failed")

merged_df <- merged_df %>%
  mutate(condition = factor(condition, levels = order_cond))

## Plot 

### C) Intensity threshold jitter

In [ ]:
library(ggplot2)
library(rlang)

plot_jitter <- function(
  data,
  x = sample_name,
  y = GATA3_norm,
  color = sample_name,
  out_dir,
  title = "",
  w = 5, h = 3,
  palette = NULL,
  hline_at = NULL,                 # <— add a dotted horizontal line at this y
  hline_color = "gray30",
  hline_size = 0.5,
  hline_lty =  "dashed"
) {
  x <- rlang::enquo(x); y <- rlang::enquo(y); color <- rlang::enquo(color)

  p <- ggplot(data, aes(x = fct_rev(!!x), y = !!y, color = !!color)) +
    geom_jitter(width = 0.2, size = 0.7, alpha = 0.6, na.rm = TRUE) +
    labs(x = "", y = "normalised intensity", title = title) +
    settheme +
    scale_y_continuous(limits = c(0, NA), expand = c(0, 0)) +
    coord_flip()

  if (!is.null(palette)) p <- p + scale_color_manual(values = palette)
  if (!is.null(hline_at)) p <- p + geom_hline(yintercept = hline_at, linetype = hline_lty,
                                              linewidth = hline_size, color = hline_color)

  ggsave(file.path(out_dir, sprintf("%s.pdf", title)), plot = p, width = w, height = h)
  p
}


### D) Intensity threshold_hist

In [ ]:
plot_hist <- function(
  data,
  x = GATA3_norm,
  facet = sample_name,
  out_dir,
  title = "GATA3_norm_hist",
  w = 5, h = 5,
  bins = 30,
  binwidth = NULL,
  fill = "#6CD1D4",
  outline = "gray10",
  linewidth = 0.2,
  free_y = TRUE,
  vline_at = NULL,              # <— vertical line at this x
  vline_color = "gray30",
  vline_size = 0.5,
  vline_lty = "dashed"
) {
  x     <- rlang::enquo(x)
  facet <- rlang::enquo(facet)

  p <- ggplot2::ggplot(data, ggplot2::aes(x = !!x)) +
    (if (!is.null(binwidth))
       ggplot2::geom_histogram(binwidth = binwidth, boundary = 0, closed = "left",
                                na.rm = TRUE, fill = fill, color = outline, linewidth = linewidth)
     else
       ggplot2::geom_histogram(bins = bins, na.rm = TRUE,
                                fill = fill, color = outline, linewidth = linewidth)) +
    ggplot2::labs(x = "normalised intensity", y = "Count", title = title) +
    settheme +
    ggplot2::scale_y_continuous(limits = c(0, NA), expand = c(0, 0),
      breaks = scales::pretty_breaks(n = 2)   # <— fewer ticks
    )  +
    ggplot2::facet_grid(rows = ggplot2::vars(!!facet),
                        scales = if (free_y) "free_y" else "fixed",
      switch = "y"            ) +
    ggplot2::theme(
      strip.text.y.left = ggplot2::element_text(angle = 0, hjust = 0),
      strip.background = ggplot2::element_blank(),
      strip.placement  = "outside",
      plot.margin = ggplot2::margin(5.5, 5.5, 5.5, 35, "pt")
    )

  if (!is.null(vline_at)) {
    p <- p + ggplot2::geom_vline(xintercept = vline_at,
                                 linetype = vline_lty,
                                 linewidth = vline_size,
                                 color = vline_color)
  }

  ggplot2::ggsave(file.path(out_dir, sprintf("%s.pdf", title)),
                  plot = p, width = w, height = h)
  p
}




In [ ]:
#dapi
w <- 4
h <- 3
options(repr.plot.width=w, repr.plot.height=h)

DAPI_thresh = 180

gghist <- plot_hist(
  data    = merged_df,
  x       = Mean_dapi,
  facet   = sample,
  out_dir = out_dir,
  binwidth = 0.1,
  title   = "D_Mean_dapi_hist",
  fill = "#362fffff",
  w = w, h = h,
  vline_at = DAPI_thresh
)
gghist

In [ ]:
w <- 5
h <- 2
options(repr.plot.width=w, repr.plot.height=h)

GATA3_thresh = GATA3_thresh

gg_jitter <- plot_jitter(
  data = merged_df,
  x = sample,
  y = GATA3_norm,
  color = sample,
  out_dir = out_dir,
  title = "C_GATA3_norm_intensity",
  w = 5, h = 3,
  #palette = col_sample,
  hline_at = GATA3_thresh
)
gg_jitter

gg_hist <- plot_hist(
  data    = merged_df,
  x       = GATA3_norm,
  facet   = sample,
  out_dir = out_dir,
  binwidth = 0.1,
  title   = "D_GATA3_norm_hist",
  fill = "#6CD1D4",
  w = 5, h = 3,
  vline_at = GATA3_thresh)
gg_hist

In [ ]:
w <- 5
h <- 2
options(repr.plot.width=w, repr.plot.height=h)

NANOG_thresh = 1.5

gg_jitter <- plot_jitter(
  data = merged_df,
  x = sample,
  y = NANOG_norm,
  color = sample,
  out_dir = out_dir,
  title = "C_NANOG_norm_intensity",
  w = 5, h = 3,
  #palette = col_condition,
  hline_at = NANOG_thresh
)
gg_jitter

gg_hist <- plot_hist(
  data    = merged_df,
  x       = NANOG_norm,
  facet   = sample,
  out_dir = out_dir,
  binwidth = 0.1,
  title   = "D_NANOG_norm_hist",
  fill = "#6CD1D4",
  w = 5, h = 3,
  vline_at = NANOG_thresh)
gg_hist

In [ ]:
w <- 5
h <- 2
options(repr.plot.width=w, repr.plot.height=h)

GATA4_thresh = GATA4_thresh

gg_jitter <- plot_jitter(
  data = merged_df,
  x = sample,
  y = GATA4_norm,
  color = sample,
  out_dir = out_dir,
  title = "C_GATA4_norm_intensity",
  w = 5, h = 3,
  #palette = col_condition,
  hline_at = GATA4_thresh
)
gg_jitter

gg_hist <- plot_hist(
  data    = merged_df,
  x       = GATA4_norm,
  facet   = sample,
  out_dir = out_dir,
  binwidth = 0.05,
  title   = "D_GATA4_norm_hist",
  fill = "#6CD1D4",
  w = 5, h = 3,
  vline_at = GATA4_thresh)
gg_hist

In [ ]:
title = "GATA3_NANOG_scatter"

w <- 6
h <- 4
options(repr.plot.width=w, repr.plot.height=h)

line_slope <- 1.4   # -0.6
line_intercept <- 1

ggscatter = ggplot(merged_df, aes(x = GATA3_norm, y = NANOG_norm, color = sample)) +
  geom_point(alpha = 0.6, size = 0.6) +
  labs(x = "GATA3_norm", y = " NANOG_norm", title = title) +
  settheme+
  geom_abline(slope = line_slope, intercept = line_intercept, linetype = "dashed") +
   #scale_color_manual(values=col_condition)+
  #geom_abline(slope = 1, intercept = 0, linetype = "dashed", color = "blue") +
  facet_wrap(~sample)+
  geom_vline(xintercept = GATA3_thresh, linetype = "dashed", color = "grey")+
  geom_hline(yintercept = NANOG_thresh, linetype = "dashed", color = "grey")



ggsave(plot = ggscatter, filename = sprintf("%s/%s.pdf",out_dir, title), w = w, h = h)
ggscatter

In [ ]:
title = "GATA3_GATA4_scatter"

w <- 6
h <- 4
options(repr.plot.width=w, repr.plot.height=h)

ggscatter = ggplot(merged_df, aes(x = GATA3_norm, y = GATA4_norm, color = sample)) +
  geom_point(alpha = 0.6, size = 0.6) +
  labs(x = "GATA3_norm", y = " GATA3_norm", title = title) +
  settheme+
   #scale_color_manual(values=col_condition)+
  #geom_abline(slope = 1, intercept = 0, linetype = "dashed", color = "blue") 
  facet_wrap(~sample)+
  geom_vline(xintercept = GATA3_thresh, linetype = "dashed", color = "grey")+
  geom_hline(yintercept = GATA4_thresh, linetype = "dashed", color = "grey")



ggsave(plot = ggscatter, filename = sprintf("%s/%s.pdf",out_dir, title), w = w, h = h)
ggscatter

### E) %pos marker

In [ ]:
plot_pct_bar_points <- function(
  data,                               # e.g., summary_df
  pct = pct_GATA3,                    # <-- column with % values to plot
  sample = sample_name,               # sample/category column
  condition = condition,              # grouping/fill column
  out_dir,                    # folder to save (optional)
  title = NULL,                       # default built from pct col name if NULL
  palette = NULL,                     # named vector for fill
  w = 2.5, h = 2,
  y_max = 110,
  y_ticks = 5,
  bar_width = 0.6,
  point_size = 0.5,
  point_alpha = 0.7,
  jitter_width = 0.05
) {
  pct      <- enquo(pct)
  sample   <- enquo(sample)
  condition<- enquo(condition)

  # default title from pct column name if not supplied
  if (is.null(title)) {
    title <- paste0(as_label(pct), "+")
  }

  # per-sample means (by condition) for the chosen pct column
  means_df <- data %>%
    group_by(!!sample, !!condition) %>%
    summarise(mean_pct = mean(!!pct, na.rm = TRUE), .groups = "drop")

  # build plot (reverse sample order, flip coords)
  p <- ggplot(data, aes(x = fct_rev(!!sample), y = !!pct)) +
    geom_col(
      data = means_df,
      aes(y = mean_pct, fill = !!condition),
      width = bar_width
    ) +
    geom_point(
      size = point_size, alpha = point_alpha,
      position = position_jitter(width = jitter_width),
      na.rm = TRUE
    ) +
    labs(x = "", y = "%", title = title, fill = rlang::as_name(condition)) +
    settheme +
    scale_y_continuous(limits = c(0, y_max), expand = c(0, 0),
                       breaks = scales::pretty_breaks(y_ticks)) +
    coord_flip()+
    theme(legend.position = "none")

  if (!is.null(palette)) {
    p <- p + scale_fill_manual(values = palette)
  }

  safe_title <- gsub("[^[:alnum:]_\\-]+","_", title)
  ggsave(file.path(out_dir, sprintf("C_%s.pdf", safe_title)),
                  plot = p, width = w, height = h)
  options(repr.plot.width=w, repr.plot.height=h)
  p
}

In [ ]:
# thresholds (edit if you want different cutoffs)
thr <- list(
  GATA3_norm = GATA3_thresh,
  GATA4_norm= GATA4_thresh,
  NANOG_norm= NANOG_thresh
  #caspase3_norm = caspase3_thresh
  #mcherry_norm= mcherry_thresh
)

summary_df <- merged_df %>%
  group_by(sample_name, condition, karyotype ,complex) %>%
  summarise(
    n = n(),
    pct_GATA3 = 100 * mean(GATA3_norm > thr$GATA3_norm & Mean_GATA3 > GATA3_thresh2, na.rm = TRUE),
    #pct_NANOG = 100 * mean(NANOG_norm > thr$NANOG_norm & GATA3_norm < thr$GATA3_norm, na.rm = TRUE),
    pct_NANOG = 100 * mean(NANOG_norm > (line_slope * GATA3_norm + line_intercept), na.rm = TRUE),
    pct_GATA4 = 100 * mean(GATA4_norm> thr$GATA4_norm, na.rm = TRUE),
    #pct_caspase3 = 100 * mean(caspase3_norm> thr$caspase3_norm, na.rm = TRUE),
    pct_negative = 100 * mean(NANOG_norm < (line_slope * GATA3_norm + line_intercept) & (GATA3_norm < thr$GATA3_norm | Mean_GATA3 < GATA3_thresh2) & GATA4_norm < thr$GATA4_norm),
    #pct_double_GFP_caspase3 = 100 * mean(GFP_norm > thr$GFP_norm & caspase3_norm > thr$caspase3_norm),
    .groups = "drop"
  )

head(summary_df)


In [ ]:
# find average intensities after subtyping
merged_df <- merged_df %>%
  mutate(
    GATA3pos = GATA3_norm > thr$GATA3_norm & Mean_GATA3 > GATA3_thresh2,
    NANOGpos = NANOG_norm > (line_slope * GATA3_norm + line_intercept),
    GATA4pos = GATA4_norm> thr$GATA4_norm
  )


In [ ]:
# Plot %GATA3+
plot_pct_bar_points(summary_df, sample = sample_name, pct = pct_GATA3, out_dir = out_dir,
                    title = "E_pctGATA3+", palette = col_condition)
# Plot %NANOG+
plot_pct_bar_points(summary_df, sample = sample_name, pct = pct_NANOG,out_dir = out_dir,
                    title = "E_pctNANOG+", palette = col_condition)
# Plot %GATA4+
plot_pct_bar_points(summary_df, sample = sample_name, pct = pct_GATA4, 
out_dir = out_dir,title = "E_pctGATA4+", palette = col_condition)


# Plot % neg
plot_pct_bar_points(summary_df, sample =sample_name,  pct = pct_negative,
out_dir = out_dir, title = "E_pctnegative", palette = col_condition)



In [ ]:
# Plot %GATA3+
plot_pct_bar_points(summary_df, sample = karyotype, pct = pct_GATA3, out_dir = out_dir,
                    title = "E_pctGATA3+", palette = col_condition)
# Plot %NANOG+
plot_pct_bar_points(summary_df, sample =  karyotype, pct = pct_NANOG,out_dir = out_dir,
                    title = "E_pctNANOG+", palette = col_condition)
# Plot %GATA4+
plot_pct_bar_points(summary_df, sample =  karyotype, pct = pct_GATA4, 
out_dir = out_dir,title = "E_pctGATA4+", palette = col_condition)


# Plot % neg
plot_pct_bar_points(summary_df, sample = karyotype,  pct = pct_negative,
out_dir = out_dir, title = "E_pctnegative", palette = col_condition)

### B) Average intensity per condition

In [ ]:
plot_pct_bar_points <- function(
  data,                               
  pct = pct_GATA3,                    
  condition = condition,              
  out_dir,                    
  title = NULL,                       
  palette = NULL,                     
  w = 1, h = 2,
  y_max = 110,
  y_ticks = 5,
  bar_width = 0.6,
  error_bar_width = 0.2,
  point_size = 0.5,
  point_alpha = 0.7,
  jitter_width = 0.1
) {
  pct       <- enquo(pct)
  condition <- enquo(condition)

  if (is.null(title)) {
    title <- paste0(as_label(pct), "+")
  }

  # 1. Calculate Summary Stats by CONDITION (Mean & SD)
  cond_summary <- data %>%
    group_by(!!condition) %>%
    summarise(
      mean_val = mean(!!pct, na.rm = TRUE),
      sd_val   = sd(!!pct, na.rm = TRUE),
      .groups  = "drop"
    )

  # 2. Build Plot
  # We map X to the Condition. 
  # Note: ensure your 'condition' column levels are set correctly before running this if you want specific order.
  p <- ggplot(data, aes(x = !!condition, y = !!pct)) +
    
    # A. The Bar (Mean of the condition)
    geom_col(
      data = cond_summary,
      aes(y = mean_val, fill = !!condition),
      width = bar_width,
      alpha = 0.5,           # Slight transparency to see points better
      show.legend = FALSE
    ) +
    
    # B. The Error Bars (Mean +/- SD)
    geom_errorbar(
      data = cond_summary,
      aes(
        y = mean_val, 
        ymin = pmax(0, mean_val - sd_val), # pmax(0, ...) prevents error bar going below 0
        ymax = mean_val + sd_val
      ),
      width = error_bar_width
    ) +
    
    # C. The Individual Points (Jittered)
    geom_jitter(
      size = point_size, 
      alpha = point_alpha,
      width = jitter_width,
      height = 0,             # Don't jitter vertically (keeps Y value accurate)
      na.rm = TRUE
    ) +
    
    labs(x = "", y = title , title = title, fill = rlang::as_name(condition)) +
    settheme +
    scale_y_continuous(limits = c(0, y_max), expand = c(0, 0),
                       breaks = scales::pretty_breaks(y_ticks)) +
      scale_fill_manual(values=col_condition)+
    theme(
      axis.text.x = element_text(angle = 45, hjust = 1),
      legend.position = "none"
    ) 

  safe_title <- gsub("[^[:alnum:]_\\-]+","_", title)
  ggsave(file.path(out_dir, sprintf("B_%s_by_condition.pdf", safe_title)),
                  plot = p, width = w, height = h)
  options(repr.plot.width=w, repr.plot.height=h)
  p
}

In [ ]:
# Define order first (optional)
summary_df$condition <- factor(summary_df$condition, levels = c("developed", "failed"))

# Plot % GATA3+
plot_pct_bar_points(summary_df, pct = pct_GATA3, palette = col_condition, out_dir = out_dir, 
                    title = "%GATA3+")

# Plot % NANOG+
plot_pct_bar_points(summary_df, pct = pct_NANOG, palette = col_condition,out_dir = out_dir,
                    title = "%NANOG+")
# Plot % GATA4+
plot_pct_bar_points(summary_df, pct = pct_GATA4, palette = col_condition,out_dir = out_dir,
                    title = "%GATA4+")

# Plot % neg
plot_pct_bar_points(summary_df, pct = pct_negative, palette = col_condition,out_dir = out_dir, 
                    title = "%negative")



In [ ]:
# Define order first (optional)
summary_df$condition <- factor(summary_df$condition, levels = c("developed", "failed"))

# Plot % GATA3+
plot_pct_bar_points(summary_df, pct = pct_GATA3, condition = complex, palette = col_condition, out_dir = out_dir, 
                    title = "%GATA3+")

# Plot % NANOG+
plot_pct_bar_points(summary_df, pct = pct_NANOG, condition = complex, palette = col_condition,out_dir = out_dir,
                    title = "%NANOG+")
# Plot % GATA4+
plot_pct_bar_points(summary_df, pct = pct_GATA4, condition = complex, palette = col_condition,out_dir = out_dir,
                    title = "%GATA4+")

# Plot % neg
plot_pct_bar_points(summary_df, pct = pct_negative, condition = complex, palette = col_condition,out_dir = out_dir, 
                    title = "%negative")



In [ ]:
plot_pct_bar_points <- function(
  data,                               
  pct = pct_GATA3,                    
  condition = condition,              
  out_dir,                    
  title = NULL,                       
  palette = NULL,                     
  w = 2, h = 2,
  y_max = 110,
  y_ticks = 5,
  bar_width = 0.6,
  error_bar_width = 0.2,
  point_size = 0.5,
  point_alpha = 0.7,
  jitter_width = 0.1
) {
  pct       <- enquo(pct)
  condition <- enquo(condition)

  if (is.null(title)) {
    title <- paste0(as_label(pct), "+")
  }

  # 1. Calculate Summary Stats by CONDITION (Mean & SD)
  cond_summary <- data %>%
    group_by(!!condition) %>%
    summarise(
      mean_val = mean(!!pct, na.rm = TRUE),
      sd_val   = sd(!!pct, na.rm = TRUE),
      .groups  = "drop"
    )

  # 2. Build Plot
  # We map X to the Condition. 
  # Note: ensure your 'condition' column levels are set correctly before running this if you want specific order.
  p <- ggplot(data, aes(x = !!condition, y = !!pct)) +
    
    # A. The Bar (Mean of the condition)
    geom_col(
      data = cond_summary,
      aes(y = mean_val, fill = !!condition),
      width = bar_width,
      alpha = 0.5,           # Slight transparency to see points better
      show.legend = FALSE
    ) +
    
    # B. The Error Bars (Mean +/- SD)
    geom_errorbar(
      data = cond_summary,
      aes(
        y = mean_val, 
        ymin = pmax(0, mean_val - sd_val), # pmax(0, ...) prevents error bar going below 0
        ymax = mean_val + sd_val
      ),
      width = error_bar_width
    ) +
    
    # C. The Individual Points (Jittered)
    geom_jitter(
      size = point_size, 
      alpha = point_alpha,
      width = jitter_width,
      height = 0,             # Don't jitter vertically (keeps Y value accurate)
      na.rm = TRUE
    ) +
    
    labs(x = "", y = title , title = title, fill = rlang::as_name(condition)) +
    settheme +
    scale_y_continuous(limits = c(0, y_max), expand = c(0, 0),
                       breaks = scales::pretty_breaks(y_ticks)) +
      scale_fill_manual(values=col_condition)+
    theme(
      axis.text.x = element_text(angle = 45, hjust = 1),
      legend.position = "none"
    ) + facet_wrap(~complex)

  safe_title <- gsub("[^[:alnum:]_\\-]+","_", title)
  ggsave(file.path(out_dir, sprintf("B_%s_by_condition.pdf", safe_title)),
                  plot = p, width = w, height = h)
  options(repr.plot.width=w, repr.plot.height=h)
  p
}

In [ ]:
# Define order first (optional)
summary_df$condition <- factor(summary_df$condition, levels = c("developed", "failed"))

# Plot % GATA3+
plot_pct_bar_points(summary_df, pct = pct_GATA3, palette = col_condition, out_dir = out_dir, 
                    title = "%GATA3+")

# Plot % NANOG+
plot_pct_bar_points(summary_df, pct = pct_NANOG, , palette = col_condition,out_dir = out_dir,
                    title = "%NANOG+")
# Plot % GATA4+
plot_pct_bar_points(summary_df, pct = pct_GATA4, palette = col_condition,out_dir = out_dir,
                    title = "%GATA4+")

# Plot % neg
plot_pct_bar_points(summary_df, pct = pct_negative, palette = col_condition,out_dir = out_dir, 
                    title = "%negative")

In [ ]:
summary_df

## Save 

In [ ]:
# 1. Calculate the average values for the "Developed" condition
ctrl_vals <- merged_df %>%
  filter(condition == "developed") %>%
  summarise(
    mean_NANOG = mean(NANOG_norm, na.rm = TRUE),
    mean_GATA3 = mean(GATA3_norm, na.rm = TRUE),
    mean_GATA4 = mean(GATA4_norm, na.rm = TRUE)
  )

# 2. Add these averages as columns and create the normalized columns
merged_df <- merged_df %>%
  mutate(
    ctrl_NANOG = ctrl_vals$mean_NANOG,
    ctrl_GATA3 = ctrl_vals$mean_GATA3,
    ctrl_GATA4 = ctrl_vals$mean_GATA4,
    
    # Create the ratio columns
    NANOG_norm_ctr = NANOG_norm / ctrl_NANOG,
    GATA3_norm_ctr = GATA3_norm / ctrl_GATA3,
    GATA4_norm_ctr = GATA4_norm / ctrl_GATA4
  )

# Check the results
head(merged_df)

## Save 

In [ ]:
merged_df$EXP = EXP
write_csv(merged_df, file.path(out_dir, "summarised_results.csv"))